# Einheit 1: Was heißt Lernen bei einer Maschine

259.075 Data-integrated Algorithmic Design Processes II

Die Aufgabe des Semesters: aus den Kennwerten eines Bestandsgebäudes den Heizwärmebedarf
vorhersagen, und zwar schnell genug, dass man beim Entwerfen hundert Varianten durchrechnet
statt einer. Heute bauen wir den einfachsten Schätzer, den es gibt.

Achtung, und das gilt für den ganzen Herbst: die Zielgröße kommt aus einer
Berechnungsvorschrift, nicht aus einer Messung. Das Modell lernt heute den Energieausweis
nach, nicht das Haus. Ab Einheit 3 kommen gemessene Daten dazu.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_csv("../data/gebaeude_platzhalter.csv")
print(df.shape)
df.head()

## Die Punktwolke

Jeder Punkt ein Haus. Waagrecht der U-Wert der Außenwand, senkrecht der Heizwärmebedarf.
Fahr mit der Maus über einen Punkt, dann siehst du Baujahr und Kennwerte.

In [ ]:
fig = px.scatter(df, x="u_wert", y="hwb", color="baujahr",
                 hover_data=["id", "baujahr", "fensteranteil", "kompaktheit"],
                 labels={"u_wert": "U-Wert [W/m2K]", "hwb": "Heizwärmebedarf [kWh/m2a]",
                         "baujahr": "Baujahr"},
                 color_continuous_scale="Viridis", height=520)
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

In [ ]:
fig = px.scatter_map(df, lat="lat", lon="lon", color="hwb", size="hwb",
                     color_continuous_scale="Turbo", zoom=10.5, height=520,
                     hover_data=["id", "baujahr", "u_wert"],
                     map_style="carto-positron")
fig.show()

## Deine eigene Gerade

Zieh am Regler. Die Gerade `hwb = w * u_wert + b` wandert mit, und oben steht, wie groß dein
Fehler ist. Such den kleinsten Wert, den du von Hand findest, und merk ihn dir.

In [ ]:
def mse(w, b, x, y):
    return float(np.mean((w * x + b - y) ** 2))

x = df["u_wert"].to_numpy()
y = df["hwb"].to_numpy()
xs = np.linspace(x.min(), x.max(), 50)
b_fix = 60.0
ws = np.arange(0, 121, 5)

frames = [go.Frame(name=f"{w}",
                   data=[go.Scatter(x=xs, y=w * xs + b_fix, mode="lines",
                                    line=dict(width=4, color="crimson"))],
                   layout=go.Layout(title=f"w = {w},  b = {b_fix:.0f},  Fehler = {mse(w, b_fix, x, y):.0f}"))
          for w in ws]

fig = go.Figure(
    data=[go.Scatter(x=x, y=y, mode="markers", name="Gebäude",
                     marker=dict(size=6, opacity=0.5, color="#4c78a8")),
          go.Scatter(x=xs, y=ws[0] * xs + b_fix, mode="lines", name="dein Modell",
                     line=dict(width=4, color="crimson"))],
    frames=frames)
fig.update_layout(
    height=560, xaxis_title="U-Wert [W/m2K]", yaxis_title="Heizwärmebedarf [kWh/m2a]",
    title=f"w = {ws[0]},  b = {b_fix:.0f},  Fehler = {mse(ws[0], b_fix, x, y):.0f}",
    sliders=[dict(active=0, currentvalue=dict(prefix="w = "),
                  steps=[dict(method="animate", label=f"{w}",
                              args=[[f"{w}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for w in ws])])
fig.show()

## Die Fehlerlandschaft

Derselbe Fehler, aber jetzt für alle Werte von `w` auf einmal aufgetragen. Das Tal ist das
Ziel. Und weil das Modell zwei Parameter hat, wird aus der Kurve eine Schale: dreh sie, dann
siehst du, wo es bergab geht.

In [ ]:
w_grid = np.linspace(0, 120, 200)
losses = [mse(w, b_fix, x, y) for w in w_grid]

fig = go.Figure(go.Scatter(x=w_grid, y=losses, mode="lines", line=dict(width=3)))
fig.update_layout(height=420, xaxis_title="w", yaxis_title="mittlerer quadratischer Fehler",
                  title=f"Fehlerkurve über w, bei festem b = {b_fix:.0f}")
fig.show()

In [ ]:
W, B = np.meshgrid(np.linspace(0, 120, 60), np.linspace(-40, 160, 60))
Z = np.array([[mse(w, b, x, y) for w in W[0]] for b in B[:, 0]])

fig = go.Figure(go.Surface(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", opacity=0.92))
fig.update_layout(height=620, title="Fehlerlandschaft über w und b",
                  scene=dict(xaxis_title="w", yaxis_title="b", zaxis_title="Fehler"))
fig.show()

## Bergab gehen

Die Steigung der Fehlerlandschaft hat einen Namen, sie heißt Gradient, und für dieses Modell
steht sie unten fertig da. Deine Aufgabe ist die Schleife: wie weit gehst du pro Schritt, und
wann hörst du auf?

Wer die Herleitung sehen will, klappt sie im Angabetext auf. Geprüft wird sie nicht.

In [ ]:
def gradient(w, b, x, y):
    fehler = w * x + b - y
    return float(2 * np.mean(fehler * x)), float(2 * np.mean(fehler))


def fit(x, y, lernrate=0.02, schritte=400, w0=0.0, b0=0.0):
    w, b = w0, b0
    verlauf = [(w, b, mse(w, b, x, y))]
    for _ in range(schritte):
        gw, gb = gradient(w, b, x, y)
        # LÜCKE: hier passiert das Lernen. Ein Schritt entgegen dem Gradienten.
        w -= lernrate * gw
        b -= lernrate * gb
        verlauf.append((w, b, mse(w, b, x, y)))
    return w, b, np.array(verlauf)


w_end, b_end, verlauf = fit(x, y)
print(f"w = {w_end:.1f}, b = {b_end:.1f}, Fehler = {verlauf[-1, 2]:.1f}")

In [ ]:
fig = go.Figure(go.Scatter(y=verlauf[:, 2], mode="lines", line=dict(width=3)))
fig.update_layout(height=380, xaxis_title="Schritt", yaxis_title="Fehler",
                  title="Verlustkurve, stabiler Lauf", yaxis_type="log")
fig.show()

## Der Abstieg als Film

Links die Höhenlinien der Fehlerlandschaft mit dem Weg, den die Parameter nehmen, rechts die
Gerade, die sich dabei in die Punktwolke legt. Play drücken.

In [ ]:
from plotly.subplots import make_subplots

schritte = range(0, 401, 8)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Weg durch die Fehlerlandschaft",
                                                    "Modell in den Daten"))
fig.add_trace(go.Contour(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", showscale=False,
                         contours=dict(showlabels=False)), row=1, col=1)
fig.add_trace(go.Scatter(x=[verlauf[0, 0]], y=[verlauf[0, 1]], mode="markers+lines",
                         marker=dict(size=10, color="crimson"), name="Parameter"), row=1, col=2 - 1)
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", marker=dict(size=5, opacity=0.4,
                         color="#4c78a8"), name="Gebäude"), row=1, col=2)
fig.add_trace(go.Scatter(x=xs, y=verlauf[0, 0] * xs + verlauf[0, 1], mode="lines",
                         line=dict(width=4, color="crimson"), name="Modell"), row=1, col=2)

fig.frames = [go.Frame(name=str(i),
                       data=[go.Scatter(x=verlauf[:i + 1, 0], y=verlauf[:i + 1, 1],
                                        mode="lines+markers",
                                        marker=dict(size=6, color="crimson")),
                             go.Scatter(x=xs, y=verlauf[i, 0] * xs + verlauf[i, 1],
                                        mode="lines", line=dict(width=4, color="crimson"))],
                       traces=[1, 3])
              for i in schritte]
fig.update_layout(height=560, showlegend=False,
                  updatemenus=[dict(type="buttons", x=0.02, y=1.15,
                                    buttons=[dict(label="Play", method="animate",
                                                  args=[None, dict(frame=dict(duration=60, redraw=True),
                                                                   fromcurrent=True)]),
                                             dict(label="Stop", method="animate",
                                                  args=[[None], dict(mode="immediate")])])])
fig.update_xaxes(title_text="w", row=1, col=1)
fig.update_yaxes(title_text="b", row=1, col=1)
fig.update_xaxes(title_text="U-Wert", row=1, col=2)
fig.update_yaxes(title_text="Heizwärmebedarf", row=1, col=2)
fig.show()

## Und jetzt kaputt machen

Dreh die Lernrate hoch, bis es explodiert. Pflicht für die Abgabe: ein stabiler und ein
divergenter Lauf, und ein Satz dazu, woran man den Unterschied in der Kurve sieht.

In [ ]:
_, _, wild = fit(x, y, lernrate=0.45, schritte=60)

fig = go.Figure(go.Scatter(y=wild[:, 2], mode="lines+markers", line=dict(width=3, color="crimson")))
fig.update_layout(height=380, xaxis_title="Schritt", yaxis_title="Fehler",
                  yaxis_type="log", title="Verlustkurve, Lernrate zu groß")
fig.show()

## Wo das Modell danebenliegt

Der Fehler bleibt hängen, und zwar nicht, weil die Schleife schlecht ist, sondern weil ein
Haus mehr hat als einen U-Wert. Fensteranteil und Kompaktheit stecken in den Daten, das
Modell kennt sie nicht, und genau dieser Rest ist der Grund für Einheit 2.

Dieselbe Karte wie am Anfang, jetzt eingefärbt nach Residuum: rot sind die Häuser, deren
Bedarf das Modell überschätzt, blau die unterschätzten. Such nach einem Muster, und sag,
woran es liegen könnte.

In [ ]:
df["vorhersage"] = w_end * x + b_end
df["residuum"] = df["hwb"] - df["vorhersage"]

fig = px.scatter_map(df, lat="lat", lon="lon", color="residuum", zoom=10.5, height=560,
                     color_continuous_scale="RdBu", range_color=(-40, 40),
                     hover_data=["id", "baujahr", "u_wert", "vorhersage", "hwb"],
                     map_style="carto-positron")
fig.show()

## Tests

Diese Zelle muss grün durchlaufen, bevor du abgibst. Es sind dieselben Tests, die auch die
Bewertung verwendet.

In [ ]:
w_t, b_t, verlauf_t = fit(x, y)
assert verlauf_t[-1, 2] < verlauf_t[0, 2], "der Fehler muss kleiner werden"
assert verlauf_t[-1, 2] < 70, "das Modell passt noch nicht gut genug"
assert np.isfinite([w_t, b_t]).all(), "Parameter sind davongelaufen"
print("alles grün")